In [ ]:
import tensorflow as tf
import pathlib
import os
from tensorflow.keras import layers

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
data_dir = pathlib.Path('/content/drive/MyDrive/dataset')

In [ ]:
batch_size = 32
img_height = 224
img_width = 224

data_augmentation = tf.keras.Sequential([
    layers.experimental.preprocessing.RandomFlip("horizontal"),
    layers.experimental.preprocessing.RandomRotation(0.1),
    layers.experimental.preprocessing.RandomZoom(0.1),
])

In [ ]:
train_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

validation_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

class_names = train_dataset.class_names

Found 4323 files belonging to 5 classes.
Using 3459 files for training.
Found 4323 files belonging to 5 classes.
Using 864 files for validation.


In [ ]:
 num_classes = len(class_names)
model = tf.keras.Sequential([
    data_augmentation,
    layers.experimental.preprocessing.Rescaling(1./255),
    layers.Conv2D(32, 3, padding='same', activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, padding='same', activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(128, 3, padding='same', activation='relu'),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(num_classes, activation='softmax')
])

In [ ]:
model.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
              metrics=['accuracy'])

In [ ]:
epochs = 35
history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=epochs
)

Epoch 1/35
109/109 [==============================] - 416s 3s/step - loss: 1.3653 - accuracy: 0.4319 - val_loss: 1.0557 - val_accuracy: 0.5532
Epoch 2/35
109/109 [==============================] - 14s 123ms/step - loss: 1.1042 - accuracy: 0.5513 - val_loss: 0.9710 - val_accuracy: 0.6111
Epoch 3/35
109/109 [==============================] - 14s 123ms/step - loss: 1.0016 - accuracy: 0.6057 - val_loss: 0.9671 - val_accuracy: 0.6053
Epoch 4/35
109/109 [==============================] - 14s 123ms/step - loss: 0.9238 - accuracy: 0.6262 - val_loss: 1.0795 - val_accuracy: 0.5995
Epoch 5/35
109/109 [==============================] - 14s 123ms/step - loss: 0.8790 - accuracy: 0.6580 - val_loss: 0.8404 - val_accuracy: 0.6644
Epoch 6/35
109/109 [==============================] - 14s 122ms/step - loss: 0.8188 - accuracy: 0.6805 - val_loss: 0.8318 - val_accuracy: 0.6759
Epoch 7/35
109/109 [==============================] - 14s 122ms/step - loss: 0.8013 - accuracy: 0.6930 - val_loss: 0.8276 - val_accu

In [ ]:
 loss, accuracy = model.evaluate(validation_dataset)
print(f"Test loss: {loss}")
print(f"Test accuracy: {accuracy}")

27/27 [==============================] - 4s 95ms/step - loss: 0.9398 - accuracy: 0.7477
Test loss: 0.9397980570793152
Test accuracy: 0.7476851940155029


AttributeError: ignored

In [ ]:
model.save('my_model.h5')

In [ ]:
import tensorflow as tf

model = tf.keras.models.load_model('/content/my_model.h5')
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()
open("model.tflite", "wb").write(tflite_model)


103143844